# C3PI/RxIMAGE Acquisition (~133k consumer-quality + ~4k reference images)

**Run this with Accelerator = None (CPU)**, via **Save Version -> Save & Run
All (Commit)** — same reasoning as `dailymed_acquisition.ipynb`: this
downloads several GB unattended and shouldn't need your GPU quota or be at
risk of an interactive-session idle timeout.

Real URLs (found by hand, not guessed — every hosting page this project
tried to fetch automatically was blocked in the authoring sandbox):
- Reference set: `https://data.lhncbc.nlm.nih.gov/public/Pills/rximage.zip`
- Consumer-quality set index: `https://data.lhncbc.nlm.nih.gov/public/Pills/index.html`

## What this notebook does, honestly

1. Downloads and extracts `rximage.zip` (the reference set), prints its
   real internal structure, then does a **best-effort** manifest build using
   the same filename-based label/domain heuristic already used for C3PI in
   the training notebook (`default_label_from_filename`, domain guessed from
   path keywords) — not yet verified against this specific zip's real
   layout, since it hasn't been downloaded in this project before.
2. Fetches the consumer-set index page and **reports every link found**
   (zip files and subdirectory links) rather than guessing how to bulk-crawl
   it — a directory-listing page for ~133k images could be a single zip, a
   per-NDC folder structure, or something else, and guessing wrong risks
   wasting a long unattended run. Paste the printed link list back so the
   bulk-download logic can be written against the real structure, the same
   way `dailymed_acquisition.ipynb`'s zip-of-zips structure was confirmed
   before writing its processing loop.


## 0. Setup

In [ ]:
import re
import zipfile
import urllib.request
from pathlib import Path

import requests

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(exist_ok=True)
C3PI_IMAGE_DIR = WORK_DIR / "c3pi_images"
C3PI_IMAGE_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = WORK_DIR / "c3pi_manifest.csv"

REFERENCE_ZIP_URL = "https://data.lhncbc.nlm.nih.gov/public/Pills/rximage.zip"
CONSUMER_INDEX_URL = "https://data.lhncbc.nlm.nih.gov/public/Pills/index.html"

print("Setup OK. Working directory:", WORK_DIR)


## 1. Download + inspect the reference set (rximage.zip)

In [ ]:
tmp_zip_path = WORK_DIR / "rximage.zip"
if not tmp_zip_path.exists():
    print(f"downloading {REFERENCE_ZIP_URL} ...")
    urllib.request.urlretrieve(REFERENCE_ZIP_URL, tmp_zip_path)
else:
    print("reusing already-downloaded rximage.zip")

print(f"Downloaded size: {tmp_zip_path.stat().st_size / 1e6:.1f} MB")

with zipfile.ZipFile(tmp_zip_path) as zf:
    names = zf.namelist()
    print(f"\nrximage.zip contains {len(names)} entries. First 30:")
    for n in names[:30]:
        print(" ", n)


## 2. Extract + build a best-effort manifest

Uses the same label/domain heuristic as the training notebook's `add_c3pi()`
(split-on-first-underscore for the label, "consumer"/"test" substring in
the path for domain) since C3PI's own reference-set naming convention has
historically followed that pattern — but this is the first time it's run
against this specific zip in this project, so **check the "first 30
entries" print above against what actually lands in the manifest below**
before trusting it for training.


In [ ]:
def default_label_from_filename(path):
    return Path(path).stem.split("_", 1)[0]

def guess_side(path):
    name = Path(path).stem.lower()
    if any(t in name for t in ("_sf", "front", "_f_", "top")):
        return "front"
    if any(t in name for t in ("_sb", "back", "_b_", "bottom")):
        return "back"
    return "unknown"

extract_dir = WORK_DIR / "rximage_extracted"
if not extract_dir.exists() or not any(extract_dir.iterdir()):
    print("extracting rximage.zip ...")
    with zipfile.ZipFile(tmp_zip_path) as zf:
        zf.extractall(extract_dir)
else:
    print("reusing already-extracted rximage/")

rows = []
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in extract_dir.rglob(ext):
        domain = "consumer" if "consumer" in str(p).lower() or "test" in str(p).lower() else "reference"
        rows.append({
            "path": str(p),
            "label": default_label_from_filename(p),
            "side": guess_side(p),
            "domain": domain,
            "source": "c3pi_reference_zip",
            "category": "RX",  # C3PI/RxIMAGE is RX-only by construction
        })

print(f"\n{len(rows)} images found in the extracted reference set")
from collections import Counter
print("By domain (heuristic — verify against the structure printed above):",
      dict(Counter(r["domain"] for r in rows)))
print(f"Distinct labels: {len(set(r['label'] for r in rows))}")

import csv
with open(MANIFEST_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["path", "label", "side", "domain", "source", "category"])
    w.writeheader()
    w.writerows(rows)
print(f"Wrote manifest to {MANIFEST_PATH}")


## 3. Consumer-quality set — report links, don't blind-download

This is the ~133k-image set (the one that actually closes the studio-vs-
real-photo gap) but its directory-listing page's real structure isn't known
yet in this project. This cell only discovers and prints what's there.


In [ ]:
resp = requests.get(CONSUMER_INDEX_URL, timeout=30)
resp.raise_for_status()

zip_links = re.findall(r'href="([^"]+\.zip)"', resp.text)
all_links = re.findall(r'href="([^"]+)"', resp.text)
dir_links = [l for l in all_links if l.endswith("/") and l not in ("../", "./")]

print(f"Found {len(zip_links)} .zip links:")
for l in zip_links[:50]:
    print(" ", l)

print(f"\nFound {len(dir_links)} subdirectory links:")
for l in dir_links[:50]:
    print(" ", l)

if not zip_links and not dir_links:
    print("\nNeither zip links nor subdirectory links found — paste the raw "
          "resp.text (or at least the <body> portion) back so the real page "
          "structure can be worked out by hand.")
elif zip_links:
    print("\nNext step: add a download loop for these zip(s), the same "
          "pattern as dailymed_acquisition.ipynb's stream_process_spl_zip — "
          "not done automatically here since the internal structure of one "
          "of these zips hasn't been confirmed yet either.")
else:
    print("\nNo direct zip — looks like a per-directory listing (likely one "
          "subfolder per NDC or per batch). Bulk-downloading this needs a "
          "recursive crawler once the structure one level in is confirmed — "
          "paste back what one of the subdirectory links contains.")
